***Movie recommendation system -- Data Cleaning***

In [1]:
#import section
import pandas as pd
from datetime import datetime, timezone

***Ratings of Movies data processing and cleaning***

In [2]:
df_ratings=pd.read_csv("H:\\CODINGS\\ML_projects\\movie_recom_system\\data\\raw\\mov_lens_small\\ratings.csv")
df_movies=pd.read_csv("H:\\CODINGS\\ML_projects\\movie_recom_system\\data\\raw\\mov_lens_small\\movies.csv")
df_links=pd.read_csv("H:\\CODINGS\\ML_projects\\movie_recom_system\\data\\raw\\mov_lens_small\\links.csv")
df_tags=pd.read_csv("H:\\CODINGS\\ML_projects\\movie_recom_system\\data\\raw\\mov_lens_small\\tags.csv")

# df_movies.head(5)

# df_links.head(5)
# df_movies.shape,df_tags.shape,df_links.shape
df_tags.drop(columns='userId',inplace=True)
df_tags.sort_values(by='movieId')

,movieId,tag,timestamp
2886,1,fun,1525286013
981,1,pixar,1137206825
629,1,pixar,1139045764
35,2,Robin Williams,1528843907
34,2,magic board game,1528843932
...,...,...,...
402,187595,star wars,1528934552
528,193565,comedy,1537098587
527,193565,anime,1537098582
530,193565,remaster,1537098592


In [3]:
#need to process the tags
# group by movie, then use join to combine tags into a single string while maintaining duplicates.
aggregated_tags = (df_tags.groupby(['movieId'])['tag'].apply(lambda x:', '.join(set(x))).reset_index())

#let's merge them all based on the movie id
merged_movie_links=(df_movies.merge(df_links, on=["movieId"], how='left').merge(aggregated_tags, on="movieId", how="left"))

#let's get the tags as well
merged_df=df_ratings.merge(merged_movie_links,on=["movieId"])

#as the timestap is not human readable , let's convert this into two different data(one will be utc and utc+6[for bd timezone])
utc_column=pd.to_datetime(df_ratings["timestamp"],utc=True,unit='s')
local_column=pd.to_datetime(df_ratings["timestamp"],utc=True,unit='s').dt.tz_convert('Asia/Dhaka')
merged_df.insert(loc=0, column="utc_column", value=utc_column)
merged_df.insert(loc=0, column="local_column", value=local_column)

# merged_df.to_csv("processed_merged_data.csv")
merged_df

,local_column,utc_column,userId,movieId,rating,timestamp,title,genres,imdbId,tmdbId,tag
0,2000-07-31 00:45:03+06:00,2000-07-30 18:45:03+00:00,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862.0,"fun, pixar"
1,2000-07-31 00:20:47+06:00,2000-07-30 18:20:47+00:00,5,1,4.0,847434962,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862.0,"fun, pixar"
2,2000-07-31 00:37:04+06:00,2000-07-30 18:37:04+00:00,7,1,4.5,1106635946,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862.0,"fun, pixar"
3,2000-07-31 01:03:35+06:00,2000-07-30 19:03:35+00:00,15,1,2.5,1510577970,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862.0,"fun, pixar"
4,2000-07-31 00:48:51+06:00,2000-07-30 18:48:51+00:00,17,1,4.5,1305696483,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862.0,"fun, pixar"
...,...,...,...,...,...,...,...,...,...,...,...
100831,2017-05-04 03:53:22+06:00,2017-05-03 21:53:22+00:00,610,160341,2.5,1479545749,Bloodmoon (1997),Action|Thriller,118745,30948.0,NaN
100832,2017-05-04 04:21:31+06:00,2017-05-03 22:21:31+00:00,610,160527,4.5,1479544998,Sympathy for the Underdog (1971),Action|Crime|Drama,66806,90351.0,NaN
100833,2017-05-09 01:50:47+06:00,2017-05-08 19:50:47+00:00,610,160836,3.0,1493844794,Hazard (2005),Action|Drama|Thriller,798722,70193.0,NaN
100834,2017-05-04 03:19:12+06:00,2017-05-03 21:19:12+00:00,610,163937,3.5,1493848789,Blair Witch (2016),Horror|Thriller,1540011,351211.0,NaN


In [4]:
#There are still some data left to preprocess like generes column data, we can make them a list.
#there are several methods to do this , i will be using the replace so that i'll not need a new column 
merged_df['genres']=merged_df['genres'].str.replace("|",",")
# merged_df.info()
# merged_df.tail()

# from the dataset info, as there are no null count less than what it should be at the tmdbid and tag, these will be our concern 
#let's check the total null value count.
print(f"Tag Column null count:{merged_df['tag'].isnull().sum()} from total row of {merged_df.shape[0]}")
print(f"tmdbId Column null count:{merged_df['tmdbId'].isnull().sum()} from total row of {merged_df.shape[0]}")

#as the tmdbid has only 13 value, we can drop those rows.
merged_df.dropna(subset=['tmdbId'], inplace=True)

#As the goal of the data to be used to give recommendation of movies.
#The system will be either content based or colleborative filtering or hybrid, where there is no need of tag.
#Another thing is that , the genres already covers the tags and the null percentage is high enough to generate noise.
#So we can remove the tag column.
merged_df.drop(columns=['tag'],inplace=True)
merged_df.info()
merged_df

Tag Column null count:52549 from total row of 100836
tmdbId Column null count:13 from total row of 100836
<class 'pandas.core.frame.DataFrame'>
Index: 100823 entries, 0 to 100835
Data columns (total 10 columns):
 #   Column        Non-Null Count   Dtype                     
---  ------        --------------   -----                     
 0   local_column  100823 non-null  datetime64[ns, Asia/Dhaka]
 1   utc_column    100823 non-null  datetime64[ns, UTC]       
 2   userId        100823 non-null  int64                     
 3   movieId       100823 non-null  int64                     
 4   rating        100823 non-null  float64                   
 5   timestamp     100823 non-null  int64                     
 6   title         100823 non-null  object                    
 7   genres        100823 non-null  object                    
 8   imdbId        100823 non-null  int64                     
 9   tmdbId        100823 non-null  float64                   
dtypes: datetime64[ns, Asia/Dhak

,local_column,utc_column,userId,movieId,rating,timestamp,title,genres,imdbId,tmdbId
0,2000-07-31 00:45:03+06:00,2000-07-30 18:45:03+00:00,1,1,4.0,964982703,Toy Story (1995),"Adventure,Animation,Children,Comedy,Fantasy",114709,862.0
1,2000-07-31 00:20:47+06:00,2000-07-30 18:20:47+00:00,5,1,4.0,847434962,Toy Story (1995),"Adventure,Animation,Children,Comedy,Fantasy",114709,862.0
2,2000-07-31 00:37:04+06:00,2000-07-30 18:37:04+00:00,7,1,4.5,1106635946,Toy Story (1995),"Adventure,Animation,Children,Comedy,Fantasy",114709,862.0
3,2000-07-31 01:03:35+06:00,2000-07-30 19:03:35+00:00,15,1,2.5,1510577970,Toy Story (1995),"Adventure,Animation,Children,Comedy,Fantasy",114709,862.0
4,2000-07-31 00:48:51+06:00,2000-07-30 18:48:51+00:00,17,1,4.5,1305696483,Toy Story (1995),"Adventure,Animation,Children,Comedy,Fantasy",114709,862.0
...,...,...,...,...,...,...,...,...,...,...
100831,2017-05-04 03:53:22+06:00,2017-05-03 21:53:22+00:00,610,160341,2.5,1479545749,Bloodmoon (1997),"Action,Thriller",118745,30948.0
100832,2017-05-04 04:21:31+06:00,2017-05-03 22:21:31+00:00,610,160527,4.5,1479544998,Sympathy for the Underdog (1971),"Action,Crime,Drama",66806,90351.0
100833,2017-05-09 01:50:47+06:00,2017-05-08 19:50:47+00:00,610,160836,3.0,1493844794,Hazard (2005),"Action,Drama,Thriller",798722,70193.0
100834,2017-05-04 03:19:12+06:00,2017-05-03 21:19:12+00:00,610,163937,3.5,1493848789,Blair Witch (2016),"Horror,Thriller",1540011,351211.0


***Now our data is almost ready, the next step will be some preprocessing to find more pattarn from the data to extract features***

*Will discuss tomorrow*